# NCHS Decision Tree Classification

**Name:** Justin Spratt
**Date:** 04/23/2026 
**Course:** CS379: Machine Learning
**Notebook Name:** `JustinSpratt CS379 IP5.ipynb`  
**Description:** This notebook implements a supervised machine learning decision tree using the **Gini impurity** splitting approach to classify the leading cause of death for records in the **NCHS - Leading Causes of Death: United States** dataset.

## Overview

This notebook implements a **DecisionTreeClassifier** that uses the **Gini impurity** splitting approach to classify the leading cause of death in the **NCHS - Leading Causes of Death: United States** dataset. I selected **Gini impurity** because this assignment is a **multiclass classification** problem rather than a regression problem. In a decision tree, Gini impurity measures how mixed the class labels are inside a node, and the algorithm selects the split that produces child nodes that are more homogeneous.

The model predicts **`Cause Name`** from the following explanatory variables:

- `Year`
- `State`
- `Deaths`
- `Age-adjusted Death Rate`

I removed the `All causes` rows before training because they represent an aggregate summary instead of a specific cause category. Leaving those rows in the target would weaken the classification task because the model would be asked to predict both detailed causes and a broad summary class.

For this notebook, the decision tree is used as an instructional example to show how a supervised machine learning workflow can be justified, implemented, evaluated, and interpreted in a single Jupyter Notebook. The dataset itself comes from the CDC's NCHS mortality data resource (Centers for Disease Control and Prevention [CDC], n.d.). The additional readings were reviewed as supplemental background. The CALIPSO article is relevant because it shows how classification rules can be represented in a decision-tree-style structure, while the structure-texture article reinforced the importance of clearly explaining algorithm design and parameter choices in technical work (Aujol et al., 2006; Kim et al., 2018).


In [ ]:
# =============================================================
# Name: Justin Spratt
# Date: 04/23/2026 
# Course: CS379: Machine Learning
# File Name: JustinSpratt CS379 IP5.ipynb
# Description: Decision tree classification using the Gini
#              impurity splitting approach on the NCHS Leading
#              Causes of Death dataset.
# =============================================================

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree


## Load the dataset

The notebook now checks several common locations for the CSV file so it runs more reliably in Jupyter. This is useful when the notebook and the dataset are not stored in exactly the same folder.


In [ ]:
# Locate the CSV file in common notebook locations.
csv_name = "NCHS_-_Leading_Causes_of_Death__United_States.csv"
possible_paths = [
    Path.cwd() / csv_name,
    Path.cwd().parent / csv_name,
    Path("/mnt/data") / csv_name,
    Path(csv_name),
]

csv_path = None
for possible_path in possible_paths:
    if possible_path.exists():
        csv_path = possible_path
        break

if csv_path is None:
    raise FileNotFoundError(
        "The dataset file was not found. Place the CSV in the same folder as "
        "the notebook or update the file path."
    )

# Read the CSV file into a pandas DataFrame.
df = pd.read_csv(csv_path)
print(f"Dataset loaded from: {csv_path}")

# Display the first few rows so the structure of the dataset is clear.
df.head()


In [ ]:
# Review the columns, shape, and missing values.
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values by column:")
print(df.isna().sum())

## Prepare the data

I kept the notebook focused on the shorter `Cause Name` column for readability. I also removed `All causes` because that row is a summary category instead of one of the specific leading causes.

For the supervised learning workflow, I used an **80/20 train-test split** with **stratification**. This means 80% of the records are used for training and 20% are reserved as unseen test data. Stratification is important here because it preserves the class distribution across both sets, which makes the evaluation more reliable and better aligned with the rubric requirement to identify and explain the test data.


In [ ]:
# Remove the aggregate category so the target contains only specific causes.
df = df[df["Cause Name"] != "All causes"].copy()

# Confirm the remaining class counts after filtering.
print("Dataset shape after removing 'All causes':", df.shape)
print("\nClass counts after filtering:")
print(df["Cause Name"].value_counts())


In [ ]:
# Select features and target.
# State is categorical, while the other selected variables are numeric.
feature_columns = ["Year", "State", "Deaths", "Age-adjusted Death Rate"]
target_column = "Cause Name"

X = df[feature_columns]
y = df[target_column]

# Split the data into training and testing sets.
# Stratification preserves the class balance in both subsets.
# This helps the test data remain representative of the full dataset.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

print("\nTraining class counts:")
print(y_train.value_counts().sort_index())

print("\nTesting class counts:")
print(y_test.value_counts().sort_index())


## Build the preprocessing and model pipeline

Because scikit-learn decision trees require numeric inputs, I one-hot encoded the `State` column. I then used a **DecisionTreeClassifier** with:

- `criterion="gini"` to apply the Gini impurity split rule
- `max_depth=8` to keep the tree interpretable and reduce overfitting
- `min_samples_leaf=5` so that leaf nodes are not too small
- `random_state=42` for reproducibility

In [ ]:
# Preprocess categorical and numeric features.
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["State"]),
        ("num", "passthrough", ["Year", "Deaths", "Age-adjusted Death Rate"]),
    ]
)

# Build a pipeline so preprocessing and modeling occur in one workflow.
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                criterion="gini",
                max_depth=8,
                min_samples_leaf=5,
                random_state=42,
            ),
        ),
    ]
)

# Train the model.
model.fit(X_train, y_train)

## Evaluate the model

This section reports both **training accuracy** and **test accuracy**, followed by the class-by-class precision, recall, and F1-score. Reporting both values is helpful because it gives a clearer picture of how well the decision tree learned from the training data and how well it generalizes to unseen test data.


In [ ]:
# Generate predictions on the training and held-out test sets.
y_train_pred = model.predict(X_train)
y_pred = model.predict(X_test)

# Evaluate the model.
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print("\nClassification Report:")
print(report)


## Illustrate the prediction output

The table below shows example predictions from the test set so the model output is easy to interpret.

In [ ]:
# Create a small table of example predictions.
results_df = X_test.reset_index(drop=True).copy()
results_df["Actual Cause"] = y_test.reset_index(drop=True)
results_df["Predicted Cause"] = y_pred

results_df.head(10)

## Feature importance

Decision trees can provide a simple feature-importance view. This helps explain which variables were most influential during splitting and supports the interpretation portion of the rubric.


In [ ]:
# Retrieve transformed feature names and feature importances.
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = pd.Series(
    model.named_steps["classifier"].feature_importances_,
    index=feature_names,
).sort_values(ascending=False)

print("Top 10 feature importances:")
print(importances.head(10))

# Plot and save the top 10 feature importances for screenshots.
fig, ax = plt.subplots(figsize=(10, 6))
importances.head(10).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 10 Feature Importances")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig("top_10_feature_importances.png", dpi=200, bbox_inches="tight")
plt.show()


## Visualize the results

These plots are useful for screenshots in the final assignment submission. I saved the figures so they can be inserted into a Word document if needed, while still keeping everything in one notebook.


In [ ]:
# Plot and save the confusion matrix.
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    xticks_rotation=90,
    colorbar=False,
    ax=ax,
)
ax.set_title("Confusion Matrix: Decision Tree with Gini Impurity")
plt.tight_layout()
plt.savefig("confusion_matrix_gini_decision_tree.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Plot and save the first few levels of the decision tree for readability.
fig, ax = plt.subplots(figsize=(26, 12))
plot_tree(
    model.named_steps["classifier"],
    feature_names=feature_names,
    class_names=model.named_steps["classifier"].classes_,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3,
    ax=ax,
)
ax.set_title("Top Levels of the Decision Tree")
plt.tight_layout()
plt.savefig("decision_tree_top_levels.png", dpi=200, bbox_inches="tight")
plt.show()


## Results explanation

The test accuracy for this notebook is in a **moderate range**, while the training accuracy is higher. That pattern is expected for a decision tree because the model can fit the training data more closely than it fits unseen data. In other words, the tree learned useful patterns from the variables, but some cause categories still overlap enough that the model makes mistakes on the test set.

A meaningful pattern in the output is that **Age-adjusted Death Rate** and **Deaths** tend to be among the most influential predictors. That makes sense because mortality rate and total deaths are strongly connected to how these leading causes differ across states and years. The feature-importance chart supports that interpretation by showing which transformed inputs had the greatest effect on splitting.

The confusion matrix adds another layer of interpretation. Some causes are separated more cleanly than others, while categories with similar mortality patterns are confused more often. That result is not unusual in a multiclass public-health dataset because multiple causes may have overlapping values across time and location.

The decision tree image is also important for the rubric because it shows the model structure directly. Even though the full tree is much larger, plotting the top levels makes the main decision rules readable. Those top nodes help explain how the tree starts grouping records into more homogeneous branches based on the selected features.


## Short discussion of algorithm choice

The selected splitting approach for this assignment is **Gini impurity**. I chose it because the target variable is categorical and the objective is to predict one class label out of several possible causes of death. Gini impurity works by favoring splits that reduce class mixing within each node, so it is an appropriate and standard choice for a multiclass classification problem.

I also reviewed the supplemental readings. The CALIPSO article is conceptually relevant because it presents classification logic through a decision-tree-style flowchart for aerosol subtypes, which mirrors the broader idea of using structured rules for classification (Kim et al., 2018). The structure-texture article is less directly related to tabular decision trees, but it still reinforced the importance of explaining algorithm selection and parameter choices clearly in technical work (Aujol et al., 2006). Those readings supported the discussion and justification portions of the assignment more than the scikit-learn implementation itself.


## References

Aujol, J.-F., Gilboa, G., Chan, T. F., & Osher, S. (2006). *Structure-texture image decompositionâ€”Modeling, algorithms, and parameter selection*. *International Journal of Computer Vision, 67*(1), 111-136. [https://doi.org/10.1007/s11263-006-4331-z](https://doi.org/10.1007/s11263-006-4331-z)

Centers for Disease Control and Prevention. (n.d.). *NCHS - Leading causes of death: United States*. Data.CDC.gov. [https://data.cdc.gov/National-Center-for-Health-Statistics/NCHS-Leading-Causes-of-Death-United-States/bi63-dtpu](https://data.cdc.gov/National-Center-for-Health-Statistics/NCHS-Leading-Causes-of-Death-United-States/bi63-dtpu)

Kim, M.-H., Omar, A. H., Tackett, J. L., Vaughan, M. A., Winker, D. M., Trepte, C. R., Hu, Y., Liu, Z., Poole, L. R., Pitts, M. C., Kar, J., & Magill, B. E. (2018). *The CALIPSO version 4 automated aerosol classification and lidar ratio selection algorithm*. *Atmospheric Measurement Techniques, 11*, 6107-6135. [https://doi.org/10.5194/amt-11-6107-2018](https://doi.org/10.5194/amt-11-6107-2018)

Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., Blondel, M., Prettenhofer, P., Weiss, R., Dubourg, V., Vanderplas, J., Passos, A., Cournapeau, D., Brucher, M., Perrot, M., & Duchesnay, Ã‰. (2011). *Scikit-learn: Machine learning in Python*. *Journal of Machine Learning Research, 12*, 2825-2830. [https://jmlr.org/papers/v12/pedregosa11a.html](https://jmlr.org/papers/v12/pedregosa11a.html)

Python Software Foundation. (n.d.). *PEP 8 â€“ Style guide for Python code*. [https://peps.python.org/pep-0008/](https://peps.python.org/pep-0008/)


## Portfolio scope and evaluation limits
This is classification of historical aggregate records, not a patient-level diagnosis or mortality forecast. Death counts and death rates are already tied to cause categories. A random row split does not evaluate generalization to future years or unseen states. Repeated state-year patterns and the national aggregate can make records dependent. The reproduced held-out accuracy was 50.51% on 1,976 rows; this is an instructional baseline, not a validated public-health model.